# XGBoost-2


## Tujuan notebook

- Notebook ini memakai basis eksperimen `Exp11`.
- Perubahan utamanya: **baseline `persistence_close` dihapus dari kandidat baseline**.
- Sisanya tetap mengikuti struktur dan setup `Exp11`: feature set, move weighting, train window 2 tahun, gate, dan evaluasi harga final.

## Cara membaca notebook ini

- **Prediksi 1** = prediksi mentah / langsung dari model.
- **Prediksi 2** = prediksi final setelah pengaman `no-harm + regime gate`. Ini adalah output utama.
- **Baseline** = model pembanding non-XGBoost yang dipilih di data validasi lalu dipakai di data test.
- Kandidat baseline di notebook ini **tidak** mencakup `persistence_close`.
- **Delta MAE** = `MAE model - MAE baseline`. Nilai **negatif** berarti model lebih baik daripada baseline.
- Rentang **`P10 / P50 / P90`** dipakai pada **Prediksi 2** untuk menunjukkan rentang prediksi final.


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    plt.style.use("seaborn-v0_8-whitegrid")

try:
    import optuna
except Exception:
    optuna = None

# Silence notebook-only tqdm widget warnings (harmless for model training).
try:
    from tqdm import TqdmWarning
    warnings.filterwarnings("ignore", category=TqdmWarning)
except Exception:
    pass

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed data" / "ali_f_event_model_ready_v3.csv"
OUT_DIR = ROOT / "data" / "processed data"

SEED = 42
H = 1
START_DATE = "2020-01-01"
DROP_LONG_GAP = True
TRAIN_WINDOW_YEARS = 2
MIN_TRAIN_ROWS = 180
MIN_VALID_ROWS = 80
MIN_TEST_ROWS = 80
MIN_TRAIN_YEARS = 2
EARLY_STOPPING = 100
INTERVAL_LOW_Q = 0.10
INTERVAL_HIGH_Q = 0.90
MOVE_WEIGHT_MIN = 0.75
MOVE_WEIGHT_MAX = 1.90
MOVE_WEIGHT_CLIP_Q = 0.90
MOVE_WEIGHT_POWER = 1.00

# No-harm gate + regime activation.
NOHARM_TAU_MULT_DEFAULT = 0.75
NOHARM_TAU_MULT_MIN = 0.20
REGIME_VOL_Z_DEFAULT = 0.75
REGIME_VOL_Z_MIN = 0.25
REGIME_VOL_Z_MAX = 2.00
REGIME_ACTIVE_TARGET_LOW = 0.20
REGIME_ACTIVE_TARGET_HIGH = 0.40

# Must-win tuning policy (valid stream must outperform locked baseline).
MUST_WIN_VALID_EPS = 1e-4
MUST_WIN_MEAN_PENALTY = 4.0
MUST_WIN_FOLD_PENALTY = 2.0
WORST_FOLD_POS_PENALTY = 3.0
REGIME_ACTIVE_PENALTY = 0.8

USE_OPTUNA = True
OPTUNA_N_TRIALS = 80
OPTUNA_TIMEOUT_SEC = 1800
OPTUNA_MIN_FOLDS = 3
OPTUNA_USE_PRUNER = True
OPTUNA_PRUNER_STARTUP_TRIALS = 12
OPTUNA_PRUNER_WARMUP_STEPS = 1

REG_PARAMS = {
    "objective": "reg:absoluteerror",
    "n_estimators": 900,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 8,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "gamma": 0.02,
    "reg_alpha": 0.8,
    "reg_lambda": 2.0,
}

REG_PARAMS_USED = REG_PARAMS.copy()
NOHARM_TAU_MULT_USED = NOHARM_TAU_MULT_DEFAULT
REGIME_VOL_Z_USED = REGIME_VOL_Z_DEFAULT
selected_params_source = "fixed_default_no_optuna"

if USE_OPTUNA and optuna is None:
    print("Optuna not available -> fallback to fixed REG_PARAMS")
    USE_OPTUNA = False

print("DATA_PATH:", DATA_PATH)
print("TARGET_HORIZON:", H)
print("REG_PARAMS(default):", REG_PARAMS)
print("NOHARM_TAU_MULT_DEFAULT:", NOHARM_TAU_MULT_DEFAULT, "| floor:", NOHARM_TAU_MULT_MIN)
print("REGIME_VOL_Z_DEFAULT:", REGIME_VOL_Z_DEFAULT)
print("MUST_WIN_VALID_EPS:", MUST_WIN_VALID_EPS)
print("USE_OPTUNA:", USE_OPTUNA, "| trials:", OPTUNA_N_TRIALS, "| timeout:", OPTUNA_TIMEOUT_SEC)
print("INTERVAL:", (INTERVAL_LOW_Q, 0.50, INTERVAL_HIGH_Q))
print("MOVE_WEIGHT_RANGE:", (MOVE_WEIGHT_MIN, MOVE_WEIGHT_MAX), "| clip_q:", MOVE_WEIGHT_CLIP_Q, "| power:", MOVE_WEIGHT_POWER)


In [ ]:
# Load data + quality context

df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

df["is_flat_ohlc"] = (
    (df["Open"] == df["High"]) & (df["High"] == df["Low"]) & (df["Low"] == df["Close"])
)

df_quality = pd.DataFrame([
    {
        "rows": len(df),
        "date_min": df["Date"].min().date(),
        "date_max": df["Date"].max().date(),
        "flat_ohlc_pct": float(df["is_flat_ohlc"].mean()),
        "long_gap_pct": float(df["is_long_gap"].fillna(False).mean()),
    }
])
df_quality


In [ ]:
# Build modeling frame (price target)

def build_price_frame(df_all: pd.DataFrame) -> pd.DataFrame:
    d = df_all.copy()
    d = d[d["Date"] >= pd.Timestamp(START_DATE)].copy()

    if DROP_LONG_GAP:
        d = d[~d["is_long_gap"].fillna(False)].copy()

    d["hl_spread_pct"] = (d["High"] - d["Low"]) / d["Close"]
    d["oc_spread_pct"] = (d["Close"] - d["Open"]) / d["Open"]
    d["volume_log1p"] = np.log1p(d["Volume"])

    for lag in (1, 2, 3, 5, 8, 10):
        d[f"ret_lag_{lag}"] = d["Return"].shift(lag)
        d[f"vol_lag_{lag}"] = d["Volume"].shift(lag)
        d[f"gap_lag_{lag}"] = d["gap_days"].shift(lag)

    d["ret_roll_mean_5"] = d["Return"].rolling(5, min_periods=3).mean()
    d["ret_roll_mean_10"] = d["Return"].rolling(10, min_periods=5).mean()
    d["ret_roll_std_10"] = d["Return"].rolling(10, min_periods=5).std()
    d["ret_roll_std_20"] = d["Return"].rolling(20, min_periods=8).std()

    d["close_mom_5"] = d["Close"].pct_change(5)
    d["close_mom_10"] = d["Close"].pct_change(10)

    d["month"] = d["Date"].dt.month
    d["quarter"] = d["Date"].dt.quarter
    d["dow"] = d["Date"].dt.dayofweek

    d[f"target_date_t{H}"] = d["Date"].shift(-H)
    d[f"target_close_t{H}"] = d["Close"].shift(-H)
    d[f"target_ret_t{H}"] = (d[f"target_close_t{H}"] / d["Close"]) - 1.0
    d[f"target_logret_t{H}"] = np.log(d[f"target_close_t{H}"] / d["Close"])

    return d


def build_yearly_walk_forward_folds(df_in: pd.DataFrame) -> list[dict]:
    years = sorted(df_in["Date"].dt.year.dropna().unique().tolist())
    folds = []

    for valid_year in years[MIN_TRAIN_YEARS:]:
        test_year = valid_year + 1
        if test_year not in years:
            continue

        train_end = pd.Timestamp(f"{valid_year - 1}-12-31")
        valid_start = pd.Timestamp(f"{valid_year}-01-01")
        valid_end = pd.Timestamp(f"{valid_year}-12-31")
        test_start = pd.Timestamp(f"{test_year}-01-01")
        test_end = pd.Timestamp(f"{test_year}-12-31")

        if TRAIN_WINDOW_YEARS is not None and int(TRAIN_WINDOW_YEARS) > 0:
            train_start = pd.Timestamp(f"{valid_year - int(TRAIN_WINDOW_YEARS)}-01-01")
            train_mask = (df_in["Date"] >= train_start) & (df_in["Date"] <= train_end)
            train_start_str = str(train_start.date())
        else:
            train_mask = df_in["Date"] <= train_end
            train_start_str = None

        folds.append(
            {
                "fold_name": f"valid_{valid_year}_test_{test_year}",
                "fold_order": len(folds) + 1,
                "train_start": train_start_str,
                "train_end": str(train_end.date()),
                "valid_start": str(valid_start.date()),
                "valid_end": str(valid_end.date()),
                "test_start": str(test_start.date()),
                "test_end": str(test_end.date()),
                "rows_train_raw": int(train_mask.sum()),
                "rows_valid_raw": int(((df_in["Date"] >= valid_start) & (df_in["Date"] <= valid_end)).sum()),
                "rows_test_raw": int(((df_in["Date"] >= test_start) & (df_in["Date"] <= test_end)).sum()),
            }
        )

    return folds


def split_by_fold_raw(df_in: pd.DataFrame, fold: dict):
    train_end = pd.Timestamp(fold["train_end"])
    train_start_raw = fold.get("train_start")
    if train_start_raw:
        train_start = pd.Timestamp(train_start_raw)
        tr = df_in[(df_in["Date"] >= train_start) & (df_in["Date"] <= train_end)].copy()
    else:
        tr = df_in[df_in["Date"] <= train_end].copy()
    va = df_in[(df_in["Date"] >= pd.Timestamp(fold["valid_start"])) & (df_in["Date"] <= pd.Timestamp(fold["valid_end"]))].copy()
    te = df_in[(df_in["Date"] >= pd.Timestamp(fold["test_start"])) & (df_in["Date"] <= pd.Timestamp(fold["test_end"]))].copy()
    return tr, va, te


def split_by_fold_purged(df_in: pd.DataFrame, fold: dict):
    td_col = f"target_date_t{H}"

    train_end = pd.Timestamp(fold["train_end"])
    train_start_raw = fold.get("train_start")
    train_start = pd.Timestamp(train_start_raw) if train_start_raw else None

    valid_start = pd.Timestamp(fold["valid_start"])
    valid_end = pd.Timestamp(fold["valid_end"])
    test_start = pd.Timestamp(fold["test_start"])
    test_end = pd.Timestamp(fold["test_end"])

    if train_start is not None:
        tr = df_in[(df_in["Date"] >= train_start) & (df_in["Date"] <= train_end) & (df_in[td_col] <= train_end)].copy()
    else:
        tr = df_in[(df_in["Date"] <= train_end) & (df_in[td_col] <= train_end)].copy()

    va = df_in[(df_in["Date"] >= valid_start) & (df_in["Date"] <= valid_end) & (df_in[td_col] <= valid_end)].copy()
    te = df_in[(df_in["Date"] >= test_start) & (df_in["Date"] <= test_end) & (df_in[td_col] <= test_end)].copy()
    return tr, va, te


FEATURES = [
    "dow", "quarter",
    "ret_lag_1", "ret_lag_5", "ret_roll_mean_5", "ret_roll_mean_10", "ret_roll_std_10", "ret_roll_std_20",
    "gap_lag_1", "hl_spread_pct", "oc_spread_pct", "close_mom_5", "close_mom_10",
]

target_info = pd.DataFrame([
    {
        "Komponen": "Target bisnis utama",
        "Nama kolom": f"target_close_t{H}",
        "Dipakai untuk": "Evaluasi hasil dan output harga final",
        "Arti": f"Harga penutupan pada t+{H}",
    },
    {
        "Komponen": "Target untuk training model",
        "Nama kolom": f"target_close_t{H} - Close",
        "Dipakai untuk": "Label yang dipelajari XGBoost",
        "Arti": f"Selisih harga target t+{H} terhadap harga saat ini",
    },
    {
        "Komponen": "Horizon target",
        "Nama kolom": f"target_date_t{H}",
        "Dipakai untuk": "Menjaga alignment tanggal target",
        "Arti": f"Tanggal observasi target pada t+{H}",
    },
])

feature_group_map = {
    "Close": "Harga / level",
    "Volume": "Volume",
    "dow": "Kalender",
    "quarter": "Kalender",
    "ret_lag_1": "Return historis",
    "ret_lag_5": "Return historis",
    "ret_roll_mean_5": "Return historis",
    "ret_roll_mean_10": "Return historis",
    "ret_roll_std_10": "Volatilitas",
    "ret_roll_std_20": "Volatilitas",
    "vol_lag_1": "Volume",
    "vol_lag_5": "Volume",
    "gap_lag_1": "Gap waktu",
    "hl_spread_pct": "Rentang harga",
    "oc_spread_pct": "Rentang harga",
    "close_mom_5": "Momentum harga",
    "close_mom_10": "Momentum harga",
}

feature_desc_map = {
    "Close": "Harga penutupan saat ini sebagai titik awal prediksi.",
    "Volume": "Volume transaksi saat ini.",
    "dow": "Hari dalam minggu untuk menangkap pola kalender.",
    "quarter": "Kuartal untuk menangkap pola musiman kasar.",
    "ret_lag_1": "Return 1 langkah ke belakang.",
    "ret_lag_5": "Return 5 langkah ke belakang.",
    "ret_roll_mean_5": "Rata-rata return jangka pendek.",
    "ret_roll_mean_10": "Rata-rata return yang sedikit lebih panjang.",
    "ret_roll_std_10": "Volatilitas return 10 langkah.",
    "ret_roll_std_20": "Volatilitas return 20 langkah.",
    "vol_lag_1": "Volume 1 langkah ke belakang.",
    "vol_lag_5": "Volume 5 langkah ke belakang.",
    "gap_lag_1": "Jarak hari ke observasi sebelumnya.",
    "hl_spread_pct": "Lebar rentang high-low relatif ke close.",
    "oc_spread_pct": "Perubahan open ke close dalam satu observasi.",
    "close_mom_5": "Momentum harga 5 langkah.",
    "close_mom_10": "Momentum harga 10 langkah.",
}

feature_info = pd.DataFrame([
    {
        "Feature": feature,
        "Kelompok": feature_group_map.get(feature, "Lainnya"),
        "Arti singkat": feature_desc_map.get(feature, "-"),
    }
    for feature in FEATURES
])

print("Target yang dipakai model")
display(target_info)

print("Daftar feature yang dipakai model")
display(feature_info)

print("Insight singkat")
print(f"- Model memakai {len(FEATURES)} feature.")
print("- Feature difokuskan ke kalender, return historis, volatilitas, gap waktu, rentang harga, dan momentum.")
print(f"- Target utama yang dibaca di output adalah harga penutupan pada t+{H}.")
print(f"- XGBoost belajar dari koreksi harga terhadap Close saat ini untuk horizon t+{H}, lalu hasilnya dijumlahkan kembali ke harga.")

feature_group_count = (
    feature_info.groupby("Kelompok", as_index=False)
    .size()
    .rename(columns={"size": "Jumlah feature"})
    .sort_values("Jumlah feature", ascending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].barh(feature_group_count["Kelompok"], feature_group_count["Jumlah feature"], color="#4c78a8")
axes[0].set_title("Jumlah Feature per Kelompok")
axes[0].set_xlabel("Jumlah feature")

axes[1].axhline(0, color="black", lw=1)
axes[1].scatter([0, H], [0, 0], s=[120, 140], color=["#7f7f7f", "#1f77b4"])
axes[1].plot([0, H], [0, 0], color="#999999", lw=2, linestyle="--")
axes[1].annotate("Informasi saat ini (t)", (0, 0), xytext=(-0.15, 0.15))
axes[1].annotate(f"Target harga di t+{H}", (H, 0), xytext=(H - 0.15, 0.15))
axes[1].set_title("Cara membaca target prediksi")
axes[1].set_xlim(-0.5, max(H + 0.5, 1.5))
axes[1].set_ylim(-0.5, 0.6)
axes[1].set_yticks([])
axes[1].set_xticks([0, H])
axes[1].set_xticklabels(["t", f"t+{H}"])

plt.tight_layout()
plt.show()

st = build_price_frame(df)
folds = build_yearly_walk_forward_folds(st)

print("Ringkasan data modeling")
print("rows in modeling frame:", len(st))
print("jumlah fold walk-forward:", len(folds))

fold_table = pd.DataFrame(folds).rename(
    columns={
        "fold_name": "Fold",
        "fold_order": "Urutan",
        "train_start": "Train mulai",
        "train_end": "Train selesai",
        "valid_start": "Valid mulai",
        "valid_end": "Valid selesai",
        "test_start": "Test mulai",
        "test_end": "Test selesai",
        "rows_train_raw": "Baris train awal",
        "rows_valid_raw": "Baris valid awal",
        "rows_test_raw": "Baris test awal",
    }
)
display(fold_table)

if not fold_table.empty:
    print("Cara baca warna: biru = train, oranye = valid, hijau = test.")

    fold_plot = pd.DataFrame(folds).copy()
    for col in ["train_start", "train_end", "valid_start", "valid_end", "test_start", "test_end"]:
        fold_plot[col] = pd.to_datetime(fold_plot[col])

    fig, ax = plt.subplots(figsize=(15, max(4.5, len(fold_plot) * 1.2)))

    for idx, row in fold_plot.iterrows():
        ax.barh(
            idx,
            (row["train_end"] - row["train_start"]).days + 1,
            left=row["train_start"],
            height=0.22,
            color="#4c78a8",
            alpha=0.95,
            label="Train" if idx == 0 else "",
        )
        ax.barh(
            idx,
            (row["valid_end"] - row["valid_start"]).days + 1,
            left=row["valid_start"],
            height=0.22,
            color="#f58518",
            alpha=0.95,
            label="Valid" if idx == 0 else "",
        )
        ax.barh(
            idx,
            (row["test_end"] - row["test_start"]).days + 1,
            left=row["test_start"],
            height=0.22,
            color="#54a24b",
            alpha=0.95,
            label="Test" if idx == 0 else "",
        )

    ax.set_yticks(np.arange(len(fold_plot)))
    ax.set_yticklabels(fold_plot["fold_name"])
    ax.invert_yaxis()
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=45)
    ax.set_xlabel("Tanggal")
    ax.set_title("Visual pembagian train / valid / test per fold")
    ax.legend(loc="upper left")

    plt.tight_layout()
    plt.show()


In [ ]:
# Leakage audit (simple but strict)

audit_rows = []
violations = []

base_cols = FEATURES + [f"target_date_t{H}", f"target_close_t{H}", f"target_logret_t{H}", "Date", "Close"]
d = st.dropna(subset=base_cols).copy().sort_values("Date").reset_index(drop=True)

if int(d["Date"].duplicated().sum()) > 0:
    violations.append({"scope": "global", "violation": "duplicate_dates"})
if int((d[f"target_date_t{H}"] <= d["Date"]).sum()) > 0:
    violations.append({"scope": "global", "violation": "target_not_strictly_future"})

for fold in folds:
    tr, va, te = split_by_fold_purged(d, fold)
    overlap_tv = int(tr["Date"].isin(va["Date"]).sum())
    overlap_tt = int(tr["Date"].isin(te["Date"]).sum())
    overlap_vt = int(va["Date"].isin(te["Date"]).sum())

    vcount = 0
    if overlap_tv > 0:
        violations.append({"scope": fold["fold_name"], "violation": "overlap_train_valid"})
        vcount += 1
    if overlap_tt > 0:
        violations.append({"scope": fold["fold_name"], "violation": "overlap_train_test"})
        vcount += 1
    if overlap_vt > 0:
        violations.append({"scope": fold["fold_name"], "violation": "overlap_valid_test"})
        vcount += 1

    if len(tr) and pd.Timestamp(tr[f"target_date_t{H}"].max()) > pd.Timestamp(fold["train_end"]):
        violations.append({"scope": fold["fold_name"], "violation": "train_target_crosses_train_end"})
        vcount += 1
    if len(va) and pd.Timestamp(va[f"target_date_t{H}"].max()) > pd.Timestamp(fold["valid_end"]):
        violations.append({"scope": fold["fold_name"], "violation": "valid_target_crosses_valid_end"})
        vcount += 1
    if len(te) and pd.Timestamp(te[f"target_date_t{H}"].max()) > pd.Timestamp(fold["test_end"]):
        violations.append({"scope": fold["fold_name"], "violation": "test_target_crosses_test_end"})
        vcount += 1

    audit_rows.append(
        {
            "fold_name": fold["fold_name"],
            "train_n": len(tr),
            "valid_n": len(va),
            "test_n": len(te),
            "violation_count": vcount,
        }
    )

audit_df = pd.DataFrame(audit_rows)
viol_df = pd.DataFrame(violations)
leak_status = "PASS" if viol_df.empty else "FAIL"

print("Leakage:", leak_status)
display(audit_df)
if not viol_df.empty:
    display(viol_df)


In [ ]:
# Train/evaluate price model

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    eps = 1e-9
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))))


def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denom = np.maximum(np.abs(y_true) + np.abs(y_pred), 1e-9)
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom))


def pinball_loss(y_true, y_pred_q, q):
    err = np.array(y_true) - np.array(y_pred_q)
    return float(np.mean(np.maximum(q * err, (q - 1.0) * err)))


def build_price_baselines(frame: pd.DataFrame, train_mean_ret: float, horizon: int) -> dict[str, np.ndarray]:
    close_t = frame["Close"].values

    drift = close_t * np.exp(train_mean_ret)
    roll5 = close_t * np.exp(horizon * frame["ret_roll_mean_5"].values)
    roll10 = close_t * np.exp(horizon * frame["ret_roll_mean_10"].values)

    mult_mom5 = np.clip(1.0 + frame["close_mom_5"].values, 0.01, None)
    repeat_last_5 = close_t * mult_mom5

    return {
        "drift_mean_ret": drift,
        "rolling_ret5_scaled": roll5,
        "rolling_ret10_scaled": roll10,
        "repeat_last_5event_move": repeat_last_5,
    }


def build_move_sample_weights(target_ret, min_w, max_w, clip_q, power):
    magnitude = np.abs(np.asarray(target_ret, dtype=float))
    if len(magnitude) == 0:
        return np.asarray([], dtype=float)
    clip_value = float(np.quantile(magnitude, float(clip_q)))
    clip_value = max(clip_value, 1e-9)
    scaled = np.clip(magnitude, 0.0, clip_value) / clip_value
    weights = float(min_w) + (float(max_w) - float(min_w)) * np.power(scaled, float(power))
    return np.asarray(weights, dtype=float)


def robust_scale(x):
    x = np.asarray(x, dtype=float)
    med = float(np.median(x))
    mad = float(np.median(np.abs(x - med)))
    if not np.isfinite(mad) or mad <= 1e-9:
        std = float(np.std(x))
        return max(std, 1.0)
    return max(1.4826 * mad, 1.0)


def robust_center_scale_ref(x_ref):
    x_ref = np.asarray(x_ref, dtype=float)
    med = float(np.median(x_ref))
    mad = float(np.median(np.abs(x_ref - med)))
    scale = 1.4826 * mad if np.isfinite(mad) and mad > 1e-9 else float(np.std(x_ref))
    return med, max(scale, 1e-6)


def build_regime_mask(vol_ref, vol_eval, z_thr):
    med, scale = robust_center_scale_ref(vol_ref)
    z = (np.asarray(vol_eval, dtype=float) - med) / scale
    return z >= float(z_thr)


def apply_regime_noharm_gate(close_t, pred_price_corr, tau_abs, regime_mask):
    close_t = np.asarray(close_t, dtype=float)
    pred_price_corr = np.asarray(pred_price_corr, dtype=float)
    regime_mask = np.asarray(regime_mask, dtype=bool)
    corr = pred_price_corr - close_t
    allow = regime_mask & (np.abs(corr) >= float(tau_abs))
    pred = np.where(allow, pred_price_corr, close_t)
    return pred, allow


def directional_metrics(pred_price, close_t, y_true_price):
    pred_dir = np.sign(np.asarray(pred_price) - np.asarray(close_t))
    true_dir = np.sign(np.asarray(y_true_price) - np.asarray(close_t))
    nz_mask = (pred_dir != 0) & (true_dir != 0)
    if nz_mask.any():
        acc_nz = float((pred_dir[nz_mask] == true_dir[nz_mask]).mean())
    else:
        acc_nz = np.nan
    return {
        "dir_acc_nonzero": acc_nz,
        "dir_nonzero_share": float(nz_mask.mean()),
    }


def build_optuna_score(mean_delta, worst_delta, std_delta, mean_active, fail_count, model_params, tau_mult, regime_z):
    base = (
        mean_delta
        + (2.20 * max(worst_delta, 0.0))
        + (0.80 * std_delta)
        + (0.02 * float(model_params.get("max_depth", 3)))
        + (0.08 * float(model_params.get("learning_rate", 0.03)))
        + (0.03 * float(max(tau_mult, NOHARM_TAU_MULT_MIN)))
        + (0.04 * float(regime_z))
    )

    must_win_pen = 0.0
    if mean_delta > -MUST_WIN_VALID_EPS:
        must_win_pen += MUST_WIN_MEAN_PENALTY
    must_win_pen += MUST_WIN_FOLD_PENALTY * float(fail_count)
    must_win_pen += WORST_FOLD_POS_PENALTY * max(worst_delta, 0.0)

    regime_pen = 0.0
    regime_pen += REGIME_ACTIVE_PENALTY * max(REGIME_ACTIVE_TARGET_LOW - mean_active, 0.0)
    regime_pen += REGIME_ACTIVE_PENALTY * max(mean_active - REGIME_ACTIVE_TARGET_HIGH, 0.0)

    return float(base + must_win_pen + regime_pen)


base_cols = FEATURES + [
    f"target_date_t{H}",
    f"target_close_t{H}",
    f"target_logret_t{H}",
    "Date",
    "Close",
    f"target_ret_t{H}",
]
d = st.dropna(subset=base_cols).copy().sort_values("Date").reset_index(drop=True)

tuning_rows = []
tuning_df = pd.DataFrame()
REG_PARAMS_USED = REG_PARAMS.copy()
NOHARM_TAU_MULT_USED = NOHARM_TAU_MULT_DEFAULT
REGIME_VOL_Z_USED = REGIME_VOL_Z_DEFAULT
selected_params_source = "fixed_default_no_optuna"

if USE_OPTUNA:
    print("Running Optuna tuning on VALID only (must-win policy, no test leakage)...")

    def _sample_model_params(trial):
        return {
            "objective": trial.suggest_categorical("objective", ["reg:absoluteerror", "reg:squarederror"]),
            "n_estimators": trial.suggest_int("n_estimators", 300, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 5),
            "min_child_weight": trial.suggest_float("min_child_weight", 4.0, 40.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 0.98),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.98),
            "gamma": trial.suggest_float("gamma", 1e-4, 8.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 20.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 50.0, log=True),
        }

    def _score_params(model_params, tau_mult, regime_z, trial=None):
        fold_deltas = []
        fold_active = []
        fail_count = 0

        for fold_idx, fold in enumerate(folds, start=1):
            tr, va, te = split_by_fold_purged(d, fold)
            if len(tr) < MIN_TRAIN_ROWS or len(va) < MIN_VALID_ROWS or len(te) < MIN_TEST_ROWS:
                continue

            Xtr = tr[FEATURES]
            Xva = va[FEATURES]

            ytr_price = tr[f"target_close_t{H}"].values
            yva_price = va[f"target_close_t{H}"].values

            ytr_corr = ytr_price - tr["Close"].values
            yva_corr = yva_price - va["Close"].values
            wtr = build_move_sample_weights(
                tr[f"target_ret_t{H}"].values,
                MOVE_WEIGHT_MIN,
                MOVE_WEIGHT_MAX,
                MOVE_WEIGHT_CLIP_Q,
                MOVE_WEIGHT_POWER,
            )

            train_mean_ret = float(tr[f"target_ret_t{H}"].mean())
            b_va = build_price_baselines(va, train_mean_ret, H)
            best_valid_base_mae = min(float(mean_absolute_error(yva_price, pred)) for pred in b_va.values())

            model = XGBRegressor(
                random_state=SEED,
                n_jobs=1,
                early_stopping_rounds=EARLY_STOPPING,
                **model_params,
            )
            model.fit(Xtr, ytr_corr, sample_weight=wtr, eval_set=[(Xva, yva_corr)], verbose=False)

            p_va_price_corr = va["Close"].values + model.predict(Xva)
            tau_abs = float(max(tau_mult, NOHARM_TAU_MULT_MIN)) * robust_scale(ytr_corr)
            reg_mask_va = build_regime_mask(tr["ret_roll_std_20"].values, va["ret_roll_std_20"].values, regime_z)
            p_va_price_noharm, _ = apply_regime_noharm_gate(va["Close"].values, p_va_price_corr, tau_abs, reg_mask_va)

            valid_mae = float(mean_absolute_error(yva_price, p_va_price_noharm))
            delta = valid_mae - best_valid_base_mae
            fold_deltas.append(delta)
            fold_active.append(float(np.mean(reg_mask_va)))
            fail_count += int(delta > -MUST_WIN_VALID_EPS)

            if trial is not None and len(fold_deltas) >= 1:
                pm = float(np.mean(fold_deltas))
                pw = float(np.max(fold_deltas))
                ps = float(np.std(fold_deltas))
                pa = float(np.mean(fold_active)) if fold_active else 0.0
                pscore = build_optuna_score(pm, pw, ps, pa, fail_count, model_params, tau_mult, regime_z)
                trial.report(pscore, step=fold_idx)
                if trial.should_prune():
                    raise optuna.TrialPruned()

        if len(fold_deltas) < OPTUNA_MIN_FOLDS:
            return 1e6, np.nan, np.nan, np.nan, np.nan, 999

        mean_delta = float(np.mean(fold_deltas))
        worst_delta = float(np.max(fold_deltas))
        std_delta = float(np.std(fold_deltas))
        mean_active = float(np.mean(fold_active)) if fold_active else 0.0

        score = build_optuna_score(
            mean_delta,
            worst_delta,
            std_delta,
            mean_active,
            fail_count,
            model_params,
            tau_mult,
            regime_z,
        )
        return float(score), mean_delta, worst_delta, std_delta, mean_active, int(fail_count)

    def _objective(trial):
        params = _sample_model_params(trial)
        tau_mult = trial.suggest_float("noharm_tau_mult", NOHARM_TAU_MULT_MIN, 2.0)
        regime_z = trial.suggest_float("regime_vol_z", REGIME_VOL_Z_MIN, REGIME_VOL_Z_MAX)

        try:
            score, mean_delta, worst_delta, std_delta, mean_active, fail_count = _score_params(
                params,
                tau_mult,
                regime_z,
                trial=trial,
            )
        except optuna.TrialPruned:
            tuning_rows.append(
                {
                    "trial": int(trial.number),
                    "state": "pruned",
                    "score": np.nan,
                    "mean_delta_valid_noharm": np.nan,
                    "worst_delta_valid_noharm": np.nan,
                    "std_delta_valid_noharm": np.nan,
                    "mean_regime_active_valid": np.nan,
                    "fail_count_valid": np.nan,
                    "noharm_tau_mult": float(tau_mult),
                    "regime_vol_z": float(regime_z),
                    **params,
                }
            )
            raise

        tuning_rows.append(
            {
                "trial": int(trial.number),
                "state": "complete",
                "score": float(score),
                "mean_delta_valid_noharm": float(mean_delta),
                "worst_delta_valid_noharm": float(worst_delta),
                "std_delta_valid_noharm": float(std_delta),
                "mean_regime_active_valid": float(mean_active),
                "fail_count_valid": int(fail_count),
                "noharm_tau_mult": float(tau_mult),
                "regime_vol_z": float(regime_z),
                **params,
            }
        )
        return float(score)

    sampler = optuna.samplers.TPESampler(seed=SEED)
    if OPTUNA_USE_PRUNER:
        pruner = optuna.pruners.MedianPruner(
            n_startup_trials=OPTUNA_PRUNER_STARTUP_TRIALS,
            n_warmup_steps=OPTUNA_PRUNER_WARMUP_STEPS,
        )
    else:
        pruner = optuna.pruners.NopPruner()

    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
    study.optimize(_objective, n_trials=OPTUNA_N_TRIALS, timeout=OPTUNA_TIMEOUT_SEC, show_progress_bar=False)

    tuning_df = pd.DataFrame(tuning_rows)
    if tuning_df.empty:
        complete_df = pd.DataFrame()
    else:
        complete_df = tuning_df[tuning_df["state"] == "complete"].copy().sort_values("score")

    # Evaluate default config under same must-win objective.
    default_score, default_mean, default_worst, default_std, default_active, default_fail_count = _score_params(
        REG_PARAMS,
        NOHARM_TAU_MULT_DEFAULT,
        REGIME_VOL_Z_DEFAULT,
        trial=None,
    )
    default_feasible = bool(
        np.isfinite(default_mean)
        and default_fail_count == 0
        and default_mean <= -MUST_WIN_VALID_EPS
    )

    feasible_df = pd.DataFrame()
    if not complete_df.empty:
        feasible_df = complete_df[
            (complete_df["fail_count_valid"] <= 0)
            & (complete_df["mean_delta_valid_noharm"] <= -MUST_WIN_VALID_EPS)
        ].copy().sort_values("score")

    if not feasible_df.empty:
        best_row = feasible_df.iloc[0]
        optuna_params = {
            "objective": str(best_row["objective"]),
            "n_estimators": int(best_row["n_estimators"]),
            "learning_rate": float(best_row["learning_rate"]),
            "max_depth": int(best_row["max_depth"]),
            "min_child_weight": float(best_row["min_child_weight"]),
            "subsample": float(best_row["subsample"]),
            "colsample_bytree": float(best_row["colsample_bytree"]),
            "gamma": float(best_row["gamma"]),
            "reg_alpha": float(best_row["reg_alpha"]),
            "reg_lambda": float(best_row["reg_lambda"]),
        }
        optuna_tau = float(best_row["noharm_tau_mult"])
        optuna_regime = float(best_row["regime_vol_z"])
        optuna_score = float(best_row["score"])

        if default_feasible and default_score <= optuna_score:
            REG_PARAMS_USED = REG_PARAMS.copy()
            NOHARM_TAU_MULT_USED = NOHARM_TAU_MULT_DEFAULT
            REGIME_VOL_Z_USED = REGIME_VOL_Z_DEFAULT
            selected_params_source = "default_feasible_fallback"
        else:
            REG_PARAMS_USED = optuna_params
            NOHARM_TAU_MULT_USED = max(optuna_tau, NOHARM_TAU_MULT_MIN)
            REGIME_VOL_Z_USED = optuna_regime
            selected_params_source = "optuna_feasible_best"
    else:
        if default_feasible:
            REG_PARAMS_USED = REG_PARAMS.copy()
            NOHARM_TAU_MULT_USED = NOHARM_TAU_MULT_DEFAULT
            REGIME_VOL_Z_USED = REGIME_VOL_Z_DEFAULT
            selected_params_source = "default_feasible_only"
        elif not complete_df.empty:
            best_row = complete_df.iloc[0]
            REG_PARAMS_USED = {
                "objective": str(best_row["objective"]),
                "n_estimators": int(best_row["n_estimators"]),
                "learning_rate": float(best_row["learning_rate"]),
                "max_depth": int(best_row["max_depth"]),
                "min_child_weight": float(best_row["min_child_weight"]),
                "subsample": float(best_row["subsample"]),
                "colsample_bytree": float(best_row["colsample_bytree"]),
                "gamma": float(best_row["gamma"]),
                "reg_alpha": float(best_row["reg_alpha"]),
                "reg_lambda": float(best_row["reg_lambda"]),
            }
            NOHARM_TAU_MULT_USED = max(float(best_row["noharm_tau_mult"]), NOHARM_TAU_MULT_MIN)
            REGIME_VOL_Z_USED = float(best_row["regime_vol_z"])
            selected_params_source = "optuna_best_nonfeasible"
        else:
            REG_PARAMS_USED = REG_PARAMS.copy()
            NOHARM_TAU_MULT_USED = NOHARM_TAU_MULT_DEFAULT
            REGIME_VOL_Z_USED = REGIME_VOL_Z_DEFAULT
            selected_params_source = "default_no_complete_trial"

    if not tuning_df.empty:
        tuning_df = tuning_df.sort_values(["state", "score"], na_position="last").reset_index(drop=True)

    print("Optuna best score:", float(study.best_value) if len(study.trials) else np.nan)
    print("Default params score:", float(default_score))
    print("Default feasible:", default_feasible, "| default_fail_count:", default_fail_count)
    print("Feasible trial count:", int(len(feasible_df)) if not feasible_df.empty else 0)
    print("Selected params source:", selected_params_source)
    print("REG_PARAMS_USED:", REG_PARAMS_USED)
    print("NOHARM_TAU_MULT_USED:", NOHARM_TAU_MULT_USED)
    print("REGIME_VOL_Z_USED:", REGIME_VOL_Z_USED)

    if not tuning_df.empty:
        tuning_overview = pd.DataFrame([
            {"Ringkasan tuning": "Trial selesai", "Nilai": int((tuning_df["state"] == "complete").sum())},
            {"Ringkasan tuning": "Trial feasible", "Nilai": int(len(feasible_df)) if not feasible_df.empty else 0},
            {"Ringkasan tuning": "Sumber parameter terpilih", "Nilai": selected_params_source},
            {"Ringkasan tuning": "Tau no-harm terpilih", "Nilai": round(float(NOHARM_TAU_MULT_USED), 4)},
            {"Ringkasan tuning": "Ambang regime terpilih", "Nilai": round(float(REGIME_VOL_Z_USED), 4)},
        ])
        print("Ringkasan tuning Optuna")
        display(tuning_overview)
else:
    print("Optuna skipped -> using fixed REG_PARAMS + NO-HARM + regime defaults")

pred_suffix = f"t{H}"
col_y_true = f"y_true_price_{pred_suffix}"
col_pred1 = f"y_pred_corr_{pred_suffix}"
col_p10 = f"y_pred_p10_{pred_suffix}"
col_pred2 = f"y_pred_p50_{pred_suffix}"
col_p90 = f"y_pred_p90_{pred_suffix}"
col_baseline = f"baseline_price_{pred_suffix}"

rows = []
pred_rows = []
skip_rows = []

for fold in folds:
    tr_raw, va_raw, te_raw = split_by_fold_raw(d, fold)
    tr, va, te = split_by_fold_purged(d, fold)

    if len(tr) < MIN_TRAIN_ROWS or len(va) < MIN_VALID_ROWS or len(te) < MIN_TEST_ROWS:
        skip_rows.append(
            {
                "fold_name": fold["fold_name"],
                "fold_order": fold["fold_order"],
                "train_n_raw": len(tr_raw),
                "valid_n_raw": len(va_raw),
                "test_n_raw": len(te_raw),
                "train_n": len(tr),
                "valid_n": len(va),
                "test_n": len(te),
                "reason": "small_split_after_purge",
            }
        )
        continue

    Xtr = tr[FEATURES]
    Xva = va[FEATURES]
    Xte = te[FEATURES]

    ytr_price = tr[f"target_close_t{H}"].values
    yva_price = va[f"target_close_t{H}"].values
    yte_price = te[f"target_close_t{H}"].values

    ytr_corr = ytr_price - tr["Close"].values
    yva_corr = yva_price - va["Close"].values
    wtr = build_move_sample_weights(
        tr[f"target_ret_t{H}"].values,
        MOVE_WEIGHT_MIN,
        MOVE_WEIGHT_MAX,
        MOVE_WEIGHT_CLIP_Q,
        MOVE_WEIGHT_POWER,
    )

    # Locked baseline selection from valid only.
    train_mean_ret = float(tr[f"target_ret_t{H}"].mean())
    b_va = build_price_baselines(va, train_mean_ret, H)
    b_te = build_price_baselines(te, train_mean_ret, H)

    base_valid = [{"name": k, "mae": float(mean_absolute_error(yva_price, v))} for k, v in b_va.items()]
    best_valid_base = sorted(base_valid, key=lambda x: x["mae"])[0]
    locked_base_name = best_valid_base["name"]

    valid_base_locked = b_va[locked_base_name]
    test_base_locked = b_te[locked_base_name]
    locked_base_test_mae = float(mean_absolute_error(yte_price, test_base_locked))

    model = XGBRegressor(
        random_state=SEED,
        n_jobs=1,
        early_stopping_rounds=EARLY_STOPPING,
        **REG_PARAMS_USED,
    )
    model.fit(Xtr, ytr_corr, sample_weight=wtr, eval_set=[(Xva, yva_corr)], verbose=False)

    # Direct correction prediction.
    p_va_price_corr = va["Close"].values + model.predict(Xva)
    p_te_price_corr = te["Close"].values + model.predict(Xte)

    # Regime-aware no-harm prediction (primary stream).
    tau_abs = float(max(NOHARM_TAU_MULT_USED, NOHARM_TAU_MULT_MIN) * robust_scale(ytr_corr))
    reg_mask_va = build_regime_mask(tr["ret_roll_std_20"].values, va["ret_roll_std_20"].values, REGIME_VOL_Z_USED)
    reg_mask_te = build_regime_mask(tr["ret_roll_std_20"].values, te["ret_roll_std_20"].values, REGIME_VOL_Z_USED)

    p_va_price_noharm, gate_applied_va = apply_regime_noharm_gate(va["Close"].values, p_va_price_corr, tau_abs, reg_mask_va)
    p_te_price_noharm, gate_applied_te = apply_regime_noharm_gate(te["Close"].values, p_te_price_corr, tau_abs, reg_mask_te)

    # Quantile range calibrated from valid residuals (primary stream).
    resid_va = yva_price - p_va_price_noharm
    res_q10 = float(np.quantile(resid_va, INTERVAL_LOW_Q))
    res_q90 = float(np.quantile(resid_va, INTERVAL_HIGH_Q))

    p_va_p10 = p_va_price_noharm + res_q10
    p_va_p90 = p_va_price_noharm + res_q90
    p_te_p10 = p_te_price_noharm + res_q10
    p_te_p90 = p_te_price_noharm + res_q90

    corr_valid_mae = float(mean_absolute_error(yva_price, p_va_price_corr))
    corr_test_mae = float(mean_absolute_error(yte_price, p_te_price_corr))

    noharm_valid_mae = float(mean_absolute_error(yva_price, p_va_price_noharm))
    noharm_test_mae = float(mean_absolute_error(yte_price, p_te_price_noharm))

    dir_m = directional_metrics(p_te_price_noharm, te["Close"].values, yte_price)

    rows.append(
        {
            "fold_name": fold["fold_name"],
            "fold_order": fold["fold_order"],
            "train_n": len(tr),
            "valid_n": len(va),
            "test_n": len(te),
            "baseline_candidate_count": len(base_valid),
            "baseline_valid_name": best_valid_base["name"],
            "baseline_valid_mae": best_valid_base["mae"],
            "baseline_test_name": locked_base_name,
            "baseline_test_mae": locked_base_test_mae,
            "xgb_corr_valid_mae": corr_valid_mae,
            "xgb_corr_test_mae": corr_test_mae,
            "xgb_noharm_valid_mae": noharm_valid_mae,
            "xgb_noharm_test_mae": noharm_test_mae,
            "delta_valid_mae_corr": corr_valid_mae - best_valid_base["mae"],
            "delta_test_mae_corr": corr_test_mae - locked_base_test_mae,
            "delta_valid_mae_noharm": noharm_valid_mae - best_valid_base["mae"],
            "delta_test_mae_noharm": noharm_test_mae - locked_base_test_mae,
            "xgb_noharm_test_rmse": float(np.sqrt(mean_squared_error(yte_price, p_te_price_noharm))),
            "xgb_noharm_test_mape": mape(yte_price, p_te_price_noharm),
            "xgb_noharm_test_smape": smape(yte_price, p_te_price_noharm),
            "xgb_noharm_test_dir_acc_nonzero": dir_m["dir_acc_nonzero"],
            "xgb_noharm_test_dir_nonzero_share": dir_m["dir_nonzero_share"],
            "noharm_test_cov80": float(((yte_price >= p_te_p10) & (yte_price <= p_te_p90)).mean()),
            "noharm_test_width80": float(np.mean(p_te_p90 - p_te_p10)),
            "noharm_test_pinball_q10": pinball_loss(yte_price, p_te_p10, INTERVAL_LOW_Q),
            "noharm_test_pinball_q90": pinball_loss(yte_price, p_te_p90, INTERVAL_HIGH_Q),
            "noharm_tau_abs": tau_abs,
            "regime_active_share_valid": float(np.mean(reg_mask_va)),
            "regime_active_share_test": float(np.mean(reg_mask_te)),
            "gate_applied_share_valid": float(np.mean(gate_applied_va)),
            "gate_applied_share_test": float(np.mean(gate_applied_te)),
        }
    )

    for split_name, split_df, y_true, p_corr, p50, p10, p90, y_base, reg_mask, gate_mask in [
        ("valid", va, yva_price, p_va_price_corr, p_va_price_noharm, p_va_p10, p_va_p90, valid_base_locked, reg_mask_va, gate_applied_va),
        ("test", te, yte_price, p_te_price_corr, p_te_price_noharm, p_te_p10, p_te_p90, test_base_locked, reg_mask_te, gate_applied_te),
    ]:
        for dt, close_t, yt, pc, p50v, p10v, p90v, yb, rm, gm in zip(
            split_df["Date"].values,
            split_df["Close"].values,
            y_true,
            p_corr,
            p50,
            p10,
            p90,
            y_base,
            reg_mask,
            gate_mask,
        ):
            pred_rows.append(
                {
                    "fold_name": fold["fold_name"],
                    "fold_order": fold["fold_order"],
                    "split": split_name,
                    "Date": pd.Timestamp(dt).strftime("%Y-%m-%d"),
                    "close_t": float(close_t),
                    col_y_true: float(yt),
                    col_pred1: float(pc),
                    col_p10: float(p10v),
                    col_pred2: float(p50v),
                    col_p90: float(p90v),
                    col_baseline: float(yb),
                    "baseline_name": locked_base_name,
                    "noharm_tau_abs": float(tau_abs),
                    "regime_active": bool(rm),
                    "gate_applied": bool(gm),
                }
            )

reg_df = pd.DataFrame(rows)
pred_df = pd.DataFrame(pred_rows)
skip_df = pd.DataFrame(skip_rows)

print("Ringkasan training dan evaluasi")
print("selected_params_source:", selected_params_source)
print("REG_PARAMS_USED:", REG_PARAMS_USED)
print("NOHARM_TAU_MULT_USED:", NOHARM_TAU_MULT_USED)
print("REGIME_VOL_Z_USED:", REGIME_VOL_Z_USED)
print("jumlah fold yang dievaluasi:", len(reg_df))
print("jumlah baris prediksi tersimpan:", len(pred_df))

fold_eval_view = reg_df[
    ["fold_name", "fold_order", "train_n", "valid_n", "test_n"]
].rename(
    columns={
        "fold_name": "Fold",
        "fold_order": "Urutan",
        "train_n": "Train setelah purge",
        "valid_n": "Valid setelah purge",
        "test_n": "Test setelah purge",
    }
)
display(fold_eval_view)

if not skip_df.empty:
    skip_view = skip_df.rename(
        columns={
            "fold_name": "Fold",
            "fold_order": "Urutan",
            "train_n_raw": "Train awal",
            "valid_n_raw": "Valid awal",
            "test_n_raw": "Test awal",
            "train_n": "Train setelah purge",
            "valid_n": "Valid setelah purge",
            "test_n": "Test setelah purge",
            "reason": "Alasan dilewati",
        }
    )
    print(f"{len(skip_view)} fold dilewati karena ukuran split terlalu kecil setelah purge.")
    display(skip_view)
else:
    print("Tidak ada fold yang dilewati setelah purge.")

baseline_catalog = pd.DataFrame([
    {"Baseline": "drift_mean_ret", "Ide singkat": "Harga bergerak mengikuti rata-rata return historis train."},
    {"Baseline": "rolling_ret5_scaled", "Ide singkat": "Harga mengikuti rata-rata return pendek terbaru."},
    {"Baseline": "rolling_ret10_scaled", "Ide singkat": "Harga mengikuti rata-rata return 10 periode terakhir."},
    {"Baseline": "repeat_last_5event_move", "Ide singkat": "Harga mengulang momentum perubahan 5 event terakhir."},
])

baseline_choice_view = reg_df[
    [
        "fold_name",
        "baseline_candidate_count",
        "baseline_valid_name",
        "baseline_valid_mae",
        "baseline_test_name",
        "baseline_test_mae",
    ]
].rename(
    columns={
        "fold_name": "Fold",
        "baseline_candidate_count": "Jumlah baseline diuji",
        "baseline_valid_name": "Baseline terbaik di valid",
        "baseline_valid_mae": "MAE baseline di valid",
        "baseline_test_name": "Baseline yang dipakai di test",
        "baseline_test_mae": "MAE baseline di test",
    }
)

baseline_count_view = (
    reg_df["baseline_test_name"]
    .value_counts()
    .rename_axis("Baseline terpilih di test")
    .reset_index(name="Jumlah fold")
)

print("Baseline yang dibandingkan")
display(baseline_catalog)

print("Baseline yang terpilih pada tiap fold")
display(baseline_choice_view)

print("Baseline yang paling sering terpilih")
display(baseline_count_view)


In [ ]:
# Ringkasan hasil utama yang mudah dibaca

if reg_df.empty:
    raise RuntimeError("No evaluated folds. Check split constraints.")

STRICT_MIN_FOLDS = 3
STRICT_MAE_EPS = 1e-4
STRICT_MAX_SIGN_FLIP = 0.34

reg_eval = reg_df.sort_values("fold_order").reset_index(drop=True).copy()
reg_eval["win_valid_noharm_strict"] = reg_eval["delta_valid_mae_noharm"] <= -STRICT_MAE_EPS
reg_eval["win_test_noharm_strict"] = reg_eval["delta_test_mae_noharm"] <= -STRICT_MAE_EPS
reg_eval["win_test_corr_strict"] = reg_eval["delta_test_mae_corr"] <= -STRICT_MAE_EPS

status_valid_noharm = np.where(reg_eval["win_valid_noharm_strict"], -1, 1)
status_test_noharm = np.where(reg_eval["win_test_noharm_strict"], -1, 1)
sign_flip_rate_noharm = float((((status_valid_noharm * status_test_noharm) == -1).mean()))

folds_ok = bool(len(reg_eval) >= STRICT_MIN_FOLDS)
all_test_folds_win_noharm = bool(reg_eval["win_test_noharm_strict"].all())
all_valid_folds_win_noharm = bool(reg_eval["win_valid_noharm_strict"].all())
robust_pass_strict_noharm = bool(
    folds_ok
    and all_test_folds_win_noharm
    and (sign_flip_rate_noharm <= STRICT_MAX_SIGN_FLIP)
)

summary = pd.DataFrame([
    {
        "folds": int(len(reg_eval)),
        "valid_win_rate_noharm_strict": float(reg_eval["win_valid_noharm_strict"].mean()),
        "test_win_rate_noharm_strict": float(reg_eval["win_test_noharm_strict"].mean()),
        "test_win_rate_corr_strict": float(reg_eval["win_test_corr_strict"].mean()),
        "mean_delta_valid_mae_noharm": float(reg_eval["delta_valid_mae_noharm"].mean()),
        "mean_delta_test_mae_noharm": float(reg_eval["delta_test_mae_noharm"].mean()),
        "worst_delta_test_mae_noharm": float(reg_eval["delta_test_mae_noharm"].max()),
        "mean_delta_test_mae_corr": float(reg_eval["delta_test_mae_corr"].mean()),
        "worst_delta_test_mae_corr": float(reg_eval["delta_test_mae_corr"].max()),
        "mean_baseline_test_mae": float(reg_eval["baseline_test_mae"].mean()),
        "mean_xgb_noharm_test_mae": float(reg_eval["xgb_noharm_test_mae"].mean()),
        "mean_xgb_corr_test_mae": float(reg_eval["xgb_corr_test_mae"].mean()),
        "mean_xgb_noharm_test_smape": float(reg_eval["xgb_noharm_test_smape"].mean()),
        "mean_xgb_noharm_test_dir_acc_nonzero": float(np.nanmean(reg_eval["xgb_noharm_test_dir_acc_nonzero"])),
        "mean_xgb_noharm_test_dir_nonzero_share": float(np.nanmean(reg_eval["xgb_noharm_test_dir_nonzero_share"])),
        "mean_noharm_test_cov80": float(reg_eval["noharm_test_cov80"].mean()),
        "mean_noharm_test_width80": float(reg_eval["noharm_test_width80"].mean()),
        "mean_noharm_tau_abs": float(reg_eval["noharm_tau_abs"].mean()),
        "mean_regime_active_share_valid": float(reg_eval["regime_active_share_valid"].mean()),
        "mean_regime_active_share_test": float(reg_eval["regime_active_share_test"].mean()),
        "mean_gate_applied_share_valid": float(reg_eval["gate_applied_share_valid"].mean()),
        "mean_gate_applied_share_test": float(reg_eval["gate_applied_share_test"].mean()),
    }
])

robust = pd.DataFrame([
    {
        "folds": int(len(reg_eval)),
        "strict_min_folds": int(STRICT_MIN_FOLDS),
        "folds_meet_min": folds_ok,
        "strict_mae_eps": float(STRICT_MAE_EPS),
        "all_test_folds_win_noharm": all_test_folds_win_noharm,
        "all_valid_folds_win_noharm": all_valid_folds_win_noharm,
        "sign_flip_rate_noharm": sign_flip_rate_noharm,
        "strict_max_sign_flip": float(STRICT_MAX_SIGN_FLIP),
        "sign_flip_ok_noharm": bool(sign_flip_rate_noharm <= STRICT_MAX_SIGN_FLIP),
        "robust_pass_strict_noharm": robust_pass_strict_noharm,
    }
])

def delta_sentence(delta_value, label):
    if delta_value < -STRICT_MAE_EPS:
        return f"{label} rata-rata lebih baik dari baseline sebesar {abs(delta_value):.4f} MAE."
    if delta_value > STRICT_MAE_EPS:
        return f"{label} rata-rata masih lebih buruk dari baseline sebesar {delta_value:.4f} MAE."
    return f"{label} rata-rata hampir sama dengan baseline."

istilah_df = pd.DataFrame([
    {"Istilah": "Prediksi 1", "Arti singkat": "Prediksi mentah langsung dari model."},
    {"Istilah": "Prediksi 2", "Arti singkat": "Prediksi final setelah pengaman; ini output utama."},
    {"Istilah": "Baseline", "Arti singkat": "Pembanding non-XGBoost untuk evaluasi; di notebook ini tanpa persistence_close."},
    {"Istilah": "Delta MAE", "Arti singkat": "MAE model dikurangi MAE baseline. Negatif = lebih baik."},
])

ringkasan_utama = pd.DataFrame([
    {
        "Pertanyaan": "Berapa fold yang benar-benar dievaluasi?",
        "Jawaban singkat": f"{len(reg_eval)} fold",
    },
    {
        "Pertanyaan": "Bagaimana baseline pada data test?",
        "Jawaban singkat": f"MAE rata-rata {reg_eval['baseline_test_mae'].mean():.4f}",
    },
    {
        "Pertanyaan": "Bagaimana Prediksi 1 pada data test?",
        "Jawaban singkat": f"MAE {reg_eval['xgb_corr_test_mae'].mean():.4f} | delta {reg_eval['delta_test_mae_corr'].mean():+.4f}",
    },
    {
        "Pertanyaan": "Bagaimana Prediksi 2 pada data test?",
        "Jawaban singkat": f"MAE {reg_eval['xgb_noharm_test_mae'].mean():.4f} | delta {reg_eval['delta_test_mae_noharm'].mean():+.4f}",
    },
    {
        "Pertanyaan": "Seberapa pas interval Prediksi 2?",
        "Jawaban singkat": f"Coverage 80% = {reg_eval['noharm_test_cov80'].mean():.2%}",
    },
    {
        "Pertanyaan": "Seberapa sering gate aktif di test?",
        "Jawaban singkat": f"{reg_eval['gate_applied_share_test'].mean():.2%} baris",
    },
])

checklist_df = pd.DataFrame([
    {"Syarat": "Leakage lulus", "Status": "Ya" if leak_status == "PASS" else "Tidak"},
    {"Syarat": "Jumlah fold minimum terpenuhi", "Status": "Ya" if folds_ok else "Tidak"},
    {"Syarat": "Semua fold valid dimenangkan Prediksi 2", "Status": "Ya" if all_valid_folds_win_noharm else "Tidak"},
    {"Syarat": "Semua fold test dimenangkan Prediksi 2", "Status": "Ya" if all_test_folds_win_noharm else "Tidak"},
    {"Syarat": "Sign flip masih dalam batas aman", "Status": "Ya" if sign_flip_rate_noharm <= STRICT_MAX_SIGN_FLIP else "Tidak"},
    {"Syarat": "Robustness ketat lulus", "Status": "Ya" if robust_pass_strict_noharm else "Tidak"},
])

ringkasan_per_fold = reg_eval[
    [
        "fold_name",
        "baseline_test_name",
        "baseline_test_mae",
        "xgb_corr_test_mae",
        "delta_test_mae_corr",
        "xgb_noharm_test_mae",
        "delta_test_mae_noharm",
        "noharm_test_cov80",
        "gate_applied_share_test",
    ]
].rename(
    columns={
        "fold_name": "Fold",
        "baseline_test_name": "Baseline di test",
        "baseline_test_mae": "MAE Baseline",
        "xgb_corr_test_mae": "MAE Prediksi 1",
        "delta_test_mae_corr": "Delta MAE Prediksi 1",
        "xgb_noharm_test_mae": "MAE Prediksi 2",
        "delta_test_mae_noharm": "Delta MAE Prediksi 2",
        "noharm_test_cov80": "Coverage 80% Prediksi 2",
        "gate_applied_share_test": "Gate aktif",
    }
).copy()

ringkasan_per_fold["Pemenang test"] = np.select(
    [
        ringkasan_per_fold["Delta MAE Prediksi 2"] < 0,
        ringkasan_per_fold["Delta MAE Prediksi 1"] < 0,
    ],
    ["Prediksi 2", "Prediksi 1"],
    default="Baseline / Imbang",
)

kolom_angka = [
    "MAE Baseline",
    "MAE Prediksi 1",
    "Delta MAE Prediksi 1",
    "MAE Prediksi 2",
    "Delta MAE Prediksi 2",
    "Coverage 80% Prediksi 2",
    "Gate aktif",
]
ringkasan_per_fold[kolom_angka] = ringkasan_per_fold[kolom_angka].round(4)

insight_lines = [
    delta_sentence(float(reg_eval["delta_test_mae_corr"].mean()), "Prediksi 1"),
    delta_sentence(float(reg_eval["delta_test_mae_noharm"].mean()), "Prediksi 2"),
    f"Coverage interval Prediksi 2 ada di {reg_eval['noharm_test_cov80'].mean():.2%}; target utamanya sekitar 80%.",
    f"Gate aktif di {reg_eval['gate_applied_share_test'].mean():.2%} baris test, jadi output final cenderung cukup konservatif.",
    "Checklist robust harus dibaca sebagai syarat lolos model, bukan sekadar statistik tambahan.",
]

print("Panduan istilah")
display(istilah_df)

print("Ringkasan utama hasil model")
display(ringkasan_utama)

print("Insight utama")
for line in insight_lines:
    print("-", line)

print("Checklist kelulusan model")
display(checklist_df)

print("Arti checklist secara high level")
print("- Leakage harus lulus agar pemisahan data aman.")
print("- Prediksi 2 harus konsisten mengalahkan atau minimal tidak kalah dari baseline di valid dan test.")
print(f"- Batas aman sign flip adalah {STRICT_MAX_SIGN_FLIP:.2f}; makin kecil biasanya makin stabil.")
print("- Jika robustness ketat tidak lulus, model belum cukup stabil untuk dianggap aman.")

print("Ringkasan hasil per fold")
display(ringkasan_per_fold)


In [ ]:
# Rincian delta MAE vs baseline, lalu visualisasi

delta_detail = reg_eval[
    [
        "fold_name",
        "baseline_test_name",
        "baseline_test_mae",
        "delta_valid_mae_noharm",
        "delta_test_mae_corr",
        "delta_test_mae_noharm",
        "xgb_corr_test_mae",
        "xgb_noharm_test_mae",
    ]
].rename(
    columns={
        "fold_name": "Fold",
        "baseline_test_name": "Baseline terkunci",
        "baseline_test_mae": "MAE Baseline (test)",
        "delta_valid_mae_noharm": "Delta MAE Prediksi 2 vs Baseline (valid)",
        "delta_test_mae_corr": "Delta MAE Prediksi 1 vs Baseline (test)",
        "delta_test_mae_noharm": "Delta MAE Prediksi 2 vs Baseline (test)",
        "xgb_corr_test_mae": "MAE Prediksi 1 (test)",
        "xgb_noharm_test_mae": "MAE Prediksi 2 (test)",
    }
).copy()

delta_detail["Interpretasi test"] = np.select(
    [
        delta_detail["Delta MAE Prediksi 2 vs Baseline (test)"] < 0,
        delta_detail["Delta MAE Prediksi 1 vs Baseline (test)"] < 0,
    ],
    [
        "Prediksi 2 lebih baik dari baseline",
        "Prediksi 1 lebih baik dari baseline",
    ],
    default="Baseline masih lebih baik / imbang",
)

kolom_delta = [
    "MAE Baseline (test)",
    "Delta MAE Prediksi 2 vs Baseline (valid)",
    "Delta MAE Prediksi 1 vs Baseline (test)",
    "Delta MAE Prediksi 2 vs Baseline (test)",
    "MAE Prediksi 1 (test)",
    "MAE Prediksi 2 (test)",
]
delta_detail[kolom_delta] = delta_detail[kolom_delta].round(4)

delta_rata2 = pd.DataFrame([
    {
        "Ringkasan": "Rata-rata delta MAE Prediksi 2 vs Baseline (valid)",
        "Nilai": round(float(reg_eval["delta_valid_mae_noharm"].mean()), 4),
    },
    {
        "Ringkasan": "Rata-rata delta MAE Prediksi 1 vs Baseline (test)",
        "Nilai": round(float(reg_eval["delta_test_mae_corr"].mean()), 4),
    },
    {
        "Ringkasan": "Rata-rata delta MAE Prediksi 2 vs Baseline (test)",
        "Nilai": round(float(reg_eval["delta_test_mae_noharm"].mean()), 4),
    },
    {
        "Ringkasan": "Delta MAE terburuk Prediksi 2 vs Baseline (test)",
        "Nilai": round(float(reg_eval["delta_test_mae_noharm"].max()), 4),
    },
])

print("Detail delta MAE terhadap baseline")
print("Catatan penting: delta MAE negatif berarti model lebih baik daripada baseline.")
display(delta_rata2)
display(delta_detail)

x = np.arange(len(reg_eval))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=False)

axes[0].bar(
    x - width / 2,
    reg_eval["delta_valid_mae_noharm"],
    width=width,
    color="#1f77b4",
    alpha=0.9,
    label="Valid - Prediksi 2",
)
axes[0].bar(
    x + width / 2,
    reg_eval["delta_test_mae_noharm"],
    width=width,
    color="#ff7f0e",
    alpha=0.9,
    label="Test - Prediksi 2",
)
axes[0].axhline(0, color="black", lw=1)
axes[0].set_xticks(x)
axes[0].set_xticklabels(reg_eval["fold_name"], rotation=25)
axes[0].set_title("Prediksi 2 vs Baseline")
axes[0].set_ylabel("Delta MAE (negatif = lebih baik)")
axes[0].legend(loc="best")

axes[1].bar(
    x - width / 2,
    reg_eval["delta_test_mae_corr"],
    width=width,
    color="#9467bd",
    alpha=0.9,
    label="Test - Prediksi 1",
)
axes[1].bar(
    x + width / 2,
    reg_eval["delta_test_mae_noharm"],
    width=width,
    color="#2ca02c",
    alpha=0.9,
    label="Test - Prediksi 2",
)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels(reg_eval["fold_name"], rotation=25)
axes[1].set_title("Prediksi 1 vs Prediksi 2 pada Data Test")
axes[1].set_ylabel("Delta MAE (negatif = lebih baik)")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()


In [ ]:
# Visual: actual vs predicted price over time (dipisah antara Prediksi 1 dan Prediksi 2)

pred_df["Date"] = pd.to_datetime(pred_df["Date"])
pred_suffix = f"t{H}"
col_y_true = f"y_true_price_{pred_suffix}"
col_pred1 = f"y_pred_corr_{pred_suffix}"
col_p10 = f"y_pred_p10_{pred_suffix}"
col_pred2 = f"y_pred_p50_{pred_suffix}"
col_p90 = f"y_pred_p90_{pred_suffix}"
col_baseline = f"baseline_price_{pred_suffix}"

def plot_prediction_stream(pred_col, label_pred, color, use_band=False):
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=False)

    for ax, split in zip(axes, ["valid", "test"]):
        d = pred_df[pred_df["split"] == split].sort_values("Date")

        ax.plot(d["Date"], d[col_y_true], color="black", lw=1.3, label=f"Harga aktual (Close t+{H})")
        ax.plot(d["Date"], d[col_baseline], color="#7f7f7f", lw=1.0, alpha=0.9, label="Baseline terkunci")

        if use_band:
            ax.fill_between(
                d["Date"],
                d[col_p10],
                d[col_p90],
                color=color,
                alpha=0.12,
                label="Rentang Prediksi 2 (P10-P90)",
            )

        ax.plot(d["Date"], d[pred_col], color=color, lw=1.2, alpha=0.95, label=label_pred)

        ax.set_title(f"{split.upper()} | Harga aktual vs {label_pred}")
        ax.set_ylabel("Price")
        ax.xaxis.set_major_locator(mdates.YearLocator(1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.tick_params(axis="x", rotation=45)
        ax.legend(loc="upper left")

    plt.tight_layout()
    plt.show()

plot_prediction_stream(
    pred_col=col_pred1,
    label_pred="Prediksi 1 (mentah langsung dari model)",
    color="#ff7f0e",
    use_band=False,
)

plot_prediction_stream(
    pred_col=col_pred2,
    label_pred="Prediksi 2 (final setelah pengaman)",
    color="#1f77b4",
    use_band=True,
)


In [ ]:
# Perbandingan performa test yang paling penting

te = pred_df[pred_df["split"] == "test"].copy()
te["Date"] = pd.to_datetime(te["Date"])
te = te.sort_values(["Date", "fold_order"]).reset_index(drop=True)

pred_suffix = f"t{H}"
col_y_true = f"y_true_price_{pred_suffix}"
col_pred1 = f"y_pred_corr_{pred_suffix}"
col_pred2 = f"y_pred_p50_{pred_suffix}"
col_baseline = f"baseline_price_{pred_suffix}"

te["resid_pred1"] = te[col_y_true] - te[col_pred1]
te["resid_pred2"] = te[col_y_true] - te[col_pred2]
te["abs_err_pred1"] = np.abs(te["resid_pred1"])
te["abs_err_pred2"] = np.abs(te["resid_pred2"])
te["abs_err_baseline"] = np.abs(te[col_y_true] - te[col_baseline])

summary_diag = pd.DataFrame([
    {
        "Versi": "Baseline",
        "MAE": float(mean_absolute_error(te[col_y_true], te[col_baseline])),
        "Delta vs Baseline": 0.0,
        "Peran": "Pembanding utama",
    },
    {
        "Versi": "Prediksi 1",
        "MAE": float(mean_absolute_error(te[col_y_true], te[col_pred1])),
        "Delta vs Baseline": float(mean_absolute_error(te[col_y_true], te[col_pred1])) - float(mean_absolute_error(te[col_y_true], te[col_baseline])),
        "Peran": "Prediksi mentah",
    },
    {
        "Versi": "Prediksi 2",
        "MAE": float(mean_absolute_error(te[col_y_true], te[col_pred2])),
        "Delta vs Baseline": float(mean_absolute_error(te[col_y_true], te[col_pred2])) - float(mean_absolute_error(te[col_y_true], te[col_baseline])),
        "Peran": "Output final",
    },
])

fold_diag = te.groupby("fold_name", as_index=False).agg(
    mae_baseline=("abs_err_baseline", "mean"),
    mae_pred1=("abs_err_pred1", "mean"),
    mae_pred2=("abs_err_pred2", "mean"),
    gate_applied_share=("gate_applied", "mean"),
)

fold_diag["winner"] = fold_diag[["mae_baseline", "mae_pred1", "mae_pred2"]].idxmin(axis=1).map(
    {
        "mae_baseline": "Baseline",
        "mae_pred1": "Prediksi 1",
        "mae_pred2": "Prediksi 2",
    }
)

fold_diag_view = fold_diag.rename(
    columns={
        "fold_name": "Fold",
        "mae_baseline": "MAE Baseline",
        "mae_pred1": "MAE Prediksi 1",
        "mae_pred2": "MAE Prediksi 2",
        "gate_applied_share": "Gate aktif",
        "winner": "Pemenang MAE",
    }
).copy()

kolom_fold = ["MAE Baseline", "MAE Prediksi 1", "MAE Prediksi 2", "Gate aktif"]
summary_diag[["MAE", "Delta vs Baseline"]] = summary_diag[["MAE", "Delta vs Baseline"]].round(4)
fold_diag_view[kolom_fold] = fold_diag_view[kolom_fold].round(4)

print("Ringkasan performa pada data test")
display(summary_diag)

print("Cara membaca bagian ini")
print("-", "MAE adalah error utama; makin kecil makin baik.")
print("-", "Delta vs Baseline negatif berarti versi tersebut lebih baik daripada baseline.")
print("-", "Kolom pemenang menunjukkan siapa yang paling kecil error-nya di tiap fold.")
print("-", "Gate aktif menunjukkan seberapa sering Prediksi 2 benar-benar dipakai sebagai output final.")

print("Metrik per fold pada data test")
display(fold_diag_view)

x = np.arange(len(fold_diag))
width = 0.25
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].bar(x - width, fold_diag["mae_baseline"], width=width, color="#7f7f7f", label="Baseline")
axes[0].bar(x, fold_diag["mae_pred1"], width=width, color="#ff7f0e", label="Prediksi 1")
axes[0].bar(x + width, fold_diag["mae_pred2"], width=width, color="#1f77b4", label="Prediksi 2")
axes[0].set_xticks(x)
axes[0].set_xticklabels(fold_diag["fold_name"], rotation=20)
axes[0].set_title("Perbandingan MAE per Fold (Test)")
axes[0].set_ylabel("MAE")
axes[0].legend(loc="best")

axes[1].bar(x, fold_diag["gate_applied_share"], width=0.4, color="#2ca02c")
axes[1].set_xticks(x)
axes[1].set_xticklabels(fold_diag["fold_name"], rotation=20)
axes[1].set_ylim(0, 1)
axes[1].set_title("Seberapa sering Gate Aktif")
axes[1].set_ylabel("Proporsi baris test")

plt.tight_layout()
plt.show()


In [ ]:
# Keputusan akhir model

leakage_pass = bool(leak_status == "PASS")

robust_row = robust.iloc[0]
folds_ok = bool(robust_row["folds_meet_min"])
all_test_folds_win_noharm = bool(robust_row["all_test_folds_win_noharm"])
all_valid_folds_win_noharm = bool(robust_row["all_valid_folds_win_noharm"])
sign_flip_ok_noharm = bool(robust_row["sign_flip_ok_noharm"])
robustness_pass_strict_noharm = bool(robust_row["robust_pass_strict_noharm"])

hard_fail_reasons = []
if not leakage_pass:
    hard_fail_reasons.append("leakage_fail")
if not folds_ok:
    hard_fail_reasons.append("insufficient_folds")
if not all_valid_folds_win_noharm:
    hard_fail_reasons.append("valid_not_outperform_locked_baseline")
if not all_test_folds_win_noharm:
    hard_fail_reasons.append("noharm_model_loses_to_locked_baseline_on_some_test_folds")
if not sign_flip_ok_noharm:
    hard_fail_reasons.append("noharm_sign_flip_rate_too_high")

overall_go = bool(leakage_pass and robustness_pass_strict_noharm)

optuna_feasible_trial_count = 0
if "tuning_df" in globals() and isinstance(tuning_df, pd.DataFrame) and not tuning_df.empty:
    if {"state", "fail_count_valid", "mean_delta_valid_noharm"}.issubset(set(tuning_df.columns)):
        optuna_feasible_trial_count = int(
            (
                (tuning_df["state"] == "complete")
                & (tuning_df["fail_count_valid"] <= 0)
                & (tuning_df["mean_delta_valid_noharm"] <= -MUST_WIN_VALID_EPS)
            ).sum()
        )

decision = {
    "target": f"close_t{H}",
    "model_type": "xgboost_feature_ablation_recent_window_with_move_weights",
    "primary_gate_stream": "xgb_noharm_price_p50",
    "output_contract": [f"price_p10_t{H}", f"price_p50_t{H}", f"price_p90_t{H}"],
    "leakage_pass": leakage_pass,
    "robustness_pass_strict_noharm": robustness_pass_strict_noharm,
    "folds_meet_min": folds_ok,
    "all_test_folds_win_noharm": all_test_folds_win_noharm,
    "all_valid_folds_win_noharm": all_valid_folds_win_noharm,
    "sign_flip_ok_noharm": sign_flip_ok_noharm,
    "overall_go_price_model": overall_go,
    "hard_fail_reasons": hard_fail_reasons,
    "mean_delta_test_mae_noharm": float(reg_eval["delta_test_mae_noharm"].mean()),
    "worst_delta_test_mae_noharm": float(reg_eval["delta_test_mae_noharm"].max()),
    "mean_delta_test_mae_corr": float(reg_eval["delta_test_mae_corr"].mean()),
    "mean_baseline_test_mae": float(reg_eval["baseline_test_mae"].mean()),
    "mean_xgb_noharm_test_mae": float(reg_eval["xgb_noharm_test_mae"].mean()),
    "mean_xgb_corr_test_mae": float(reg_eval["xgb_corr_test_mae"].mean()),
    "mean_xgb_noharm_test_smape": float(reg_eval["xgb_noharm_test_smape"].mean()),
    "mean_xgb_noharm_test_dir_acc_nonzero": float(np.nanmean(reg_eval["xgb_noharm_test_dir_acc_nonzero"])),
    "mean_xgb_noharm_test_dir_nonzero_share": float(np.nanmean(reg_eval["xgb_noharm_test_dir_nonzero_share"])),
    "mean_noharm_test_cov80": float(reg_eval["noharm_test_cov80"].mean()),
    "mean_regime_active_share_test": float(reg_eval["regime_active_share_test"].mean()),
    "mean_gate_applied_share_test": float(reg_eval["gate_applied_share_test"].mean()),
    "feature_count": int(len(FEATURES)),
    "selected_params_source": selected_params_source,
    "optuna_feasible_trial_count": int(optuna_feasible_trial_count),
    "reg_params_used": REG_PARAMS_USED,
    "noharm_tau_mult_used": float(NOHARM_TAU_MULT_USED),
    "noharm_tau_mult_min": float(NOHARM_TAU_MULT_MIN),
    "regime_vol_z_used": float(REGIME_VOL_Z_USED),
    "regime_active_target_low": float(REGIME_ACTIVE_TARGET_LOW),
    "regime_active_target_high": float(REGIME_ACTIVE_TARGET_HIGH),
    "use_optuna": bool(USE_OPTUNA),
    "optuna_trials": int(OPTUNA_N_TRIALS) if USE_OPTUNA else 0,
}

decision_view = pd.DataFrame([
    {"Poin": "Model layak dipakai?", "Nilai": "Ya" if overall_go else "Tidak"},
    {"Poin": "Output utama", "Nilai": "Prediksi 2"},
    {"Poin": "Leakage lulus?", "Nilai": "Ya" if leakage_pass else "Tidak"},
    {"Poin": "Semua fold valid dimenangkan Prediksi 2?", "Nilai": "Ya" if all_valid_folds_win_noharm else "Tidak"},
    {"Poin": "Semua fold test dimenangkan Prediksi 2?", "Nilai": "Ya" if all_test_folds_win_noharm else "Tidak"},
    {"Poin": "Alasan gagal utama", "Nilai": ", ".join(hard_fail_reasons) if hard_fail_reasons else "Tidak ada"},
    {"Poin": "Sumber parameter", "Nilai": selected_params_source},
])

params_view = pd.DataFrame(
    [{"Parameter": key, "Nilai": value} for key, value in REG_PARAMS_USED.items()]
)

print("Keputusan akhir model")
display(decision_view)

print("Cara membaca keputusan")
print("- Jika model belum layak dipakai, lihat kolom alasan gagal utama terlebih dahulu.")
print("- Output utama yang dipakai tetap Prediksi 2, bukan Prediksi 1.")
print("- Sumber parameter membantu melihat apakah model memakai default atau hasil tuning.")

print("Parameter XGBoost yang dipakai")
display(params_view)


In [ ]:
# Save artifacts
SAVE_ARTIFACTS = False
prefix = 'xgb_2_exp11_no_persistence_close_h1'

if SAVE_ARTIFACTS:
    reg_eval.to_csv(OUT_DIR / f"{prefix}_fold_results.csv", index=False)
    pred_df.to_csv(OUT_DIR / f"{prefix}_predictions.csv", index=False)
    audit_df.to_csv(OUT_DIR / f"{prefix}_leakage_audit.csv", index=False)
    viol_df.to_csv(OUT_DIR / f"{prefix}_leakage_violations.csv", index=False)
    summary.to_csv(OUT_DIR / f"{prefix}_summary.csv", index=False)
    robust.to_csv(OUT_DIR / f"{prefix}_robustness.csv", index=False)
    if "tuning_df" in globals() and isinstance(tuning_df, pd.DataFrame) and not tuning_df.empty:
        tuning_df.to_csv(OUT_DIR / f"{prefix}_optuna_trials.csv", index=False)
    (OUT_DIR / f"{prefix}_decision.json").write_text(json.dumps(decision, indent=2))
    print("saved artifacts with prefix:", prefix)
else:
    print("SAVE_ARTIFACTS=False -> artifacts not written")
